## 0. Setup & imports

In [ ]:
%matplotlib inline
import os, re, sys, glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, FixedLocator, NullLocator

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix,
                             cohen_kappa_score, average_precision_score)
from scipy.stats import randint, uniform, loguniform
from xgboost import XGBClassifier
import statsmodels.formula.api as smf


SEEDS = [4, 8, 9, 10, 30, 64, 67, 88, 117, 149]   # 10 seeds
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/ctabgan_repeatability"
except Exception as _e:
    print("Drive not mounted (not in Colab?) - using local ./artifacts :", _e)
    DRIVE_ROOT = "."
BASE = os.path.join(DRIVE_ROOT, "artifacts")   # shared inputs + every seed's outputs live here
os.makedirs(BASE, exist_ok=True)
print("Outputs ->", BASE)

CTABGAN_REPO = os.environ.get("CTABGAN_REPO", "CTAB-GAN")
print("Setup complete. Seeds:", SEEDS)


In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install xgboost statsmodels imbalanced-learn torch
    if not os.path.isdir(CTABGAN_REPO):
        !git clone -q https://github.com/Team-TUD/CTAB-GAN.git
    assert os.path.exists("diabetic_data.csv"), \
        "Upload diabetic_data.csv to the Colab file panel."
print("bootstrap done | CTAB-GAN repo:", CTABGAN_REPO)


## 1. Stage 1 - cleaning & feature engineering
Produces `df_model` (features + `readmit_class`).

In [ ]:
df = pd.read_csv('diabetic_data.csv')
print("raw shape:", df.shape)

# Remove encounters ending in death/hospice (cannot be readmitted)
df = df.replace('?', np.nan)
df = df[~df['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])].copy()

# Drop duplicate patients (keep first)
if {'patient_nbr', 'encounter_id'}.issubset(df.columns):
    df = df.sort_values(['patient_nbr', 'encounter_id'])
    df = df.drop_duplicates(subset='patient_nbr', keep='first')

# Drop unwanted columns
df.drop(columns=['weight', 'payer_code', 'encounter_id', 'patient_nbr',
                 'examide', 'citoglipton'], inplace=True)
print("after cleaning:", df.shape)

# ICD-9 diagnosis category flags
def map_icd9_category(code):
    if pd.isna(code):
        return np.nan
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    try:
        code_num = float(code)
    except ValueError:
        return "Other"
    if 250 <= code_num < 251:            return "Diabetes"
    if (390 <= code_num <= 459) or code_num == 785:  return "Circulatory"
    if (460 <= code_num <= 519) or code_num == 786:  return "Respiratory"
    if (520 <= code_num <= 579) or code_num == 787:  return "Digestive"
    if 800 <= code_num <= 999:           return "Injury"
    if 710 <= code_num <= 739:           return "Musculoskeletal"
    if (580 <= code_num <= 629) or code_num == 788:  return "Genitourinary"
    if 140 <= code_num <= 239:           return "Cancer"
    return "Other"

diag_raw_cols = [c for c in ["diag_1", "diag_2", "diag_3"] if c in df.columns]
cats = df[diag_raw_cols].applymap(map_icd9_category)
diag_categories = ["Circulatory", "Respiratory", "Digestive", "Diabetes",
                   "Injury", "Musculoskeletal", "Genitourinary", "Cancer", "Other"]
df["n_distinct_diag_categories"] = cats.nunique(axis=1)
df["diabetes_primary_diagnosis"] = (cats["diag_1"] == "Diabetes").astype(int)
for category in diag_categories:
    df[f"has_{category.lower()}"] = (cats == category).any(axis=1).astype(int)
df.drop(columns=diag_raw_cols, inplace=True)

# Medical specialty top-10 + other
if "medical_specialty" in df.columns:
    spec = df["medical_specialty"].replace("?", np.nan)
    top10 = spec.value_counts().head(10).index
    slug = lambda x: re.sub(r"[^0-9a-zA-Z]+", "_", str(x)).strip("_")
    for s in top10:
        df[f"spec_{slug(s)}"] = (spec == s).astype(int)
    df["spec_other"] = (~spec.isin(top10)).astype(int)
    df.drop(columns="medical_specialty", inplace=True)

# Age midpoint
age_map = {"[0-10)":5,"[10-20)":15,"[20-30)":25,"[30-40)":35,"[40-50)":45,
           "[50-60)":55,"[60-70)":65,"[70-80)":75,"[80-90)":85,"[90-100)":95}
df["age_numeric"] = df["age"].map(age_map)

# Medication counts
med_status_cols = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride",
    "acetohexamide","glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone",
    "acarbose","miglitol","troglitazone","tolazamide","glyburide-metformin","glipizide-metformin",
    "glimepiride-pioglitazone","metformin-rosiglitazone","metformin-pioglitazone","insulin"]
med_cols_present = [c for c in med_status_cols if c in df.columns]
df["diabetes_med_count"] = (df[med_cols_present] != "No").sum(axis=1)
df["total_med_changes"] = (df[med_cols_present].isin(['Up', 'Down'])).sum(axis=1)

# Visit / procedure aggregates
df["total_visits"] = (df["number_outpatient"].fillna(0) + df["number_emergency"].fillna(0)
                      + df["number_inpatient"].fillna(0))
df['sum_emergency_and_inpatient'] = (df['number_emergency'].fillna(0) + df['number_inpatient'].fillna(0))
df["total_procedures"] = (df["num_lab_procedures"].fillna(0) + df["num_procedures"].fillna(0))
df['medical_burden'] = (df['total_procedures'].fillna(0) + df['number_diagnoses'].fillna(0))
df['many_medications'] = (df['num_medications'] >= 5).astype(int)
df['long_stay'] = (df['time_in_hospital'] >= 7).astype(int)

# A1C / glucose lab features
df["a1c_measured"] = df["A1Cresult"].notna().astype(int)
df["glu_measured"] = df["max_glu_serum"].notna().astype(int)
def map_a1c_level(x):
    if pd.isna(x) or x == "None": return 0
    if x == "Norm": return 1
    if x == ">7": return 2
    if x == ">8": return 3
    return 0
df["a1c_level"] = df["A1Cresult"].apply(map_a1c_level)
def map_glu_level(x):
    if pd.isna(x) or x == "None": return 0
    if x == "Norm": return 1
    if x == ">200": return 2
    if x == ">300": return 3
    return 0
df["glu_level"] = df["max_glu_serum"].apply(map_glu_level)
df["a1c_high"] = df["A1Cresult"].isin([">7", ">8"]).astype(int)
df["glu_high"] = df["max_glu_serum"].isin([">200", ">300"]).astype(int)

# Admission / discharge context
df["is_emergency_admission"] = df["admission_type_id"].isin({1, 7}).astype(int)
df["is_unknown_discharge"] = df["discharge_disposition_id"].isin({9, 18, 25, 26}).astype(int)
df["discharged_to_another_facility"] = (~df["discharge_disposition_id"].isin({1, 6, 8, 9, 18, 25, 26})).astype(int)
df["admitted_from_er"] = df["admission_source_id"].isin({7}).astype(int)

# Race / gender indicators
race_clean = df["race"].replace("?", np.nan)
for r in race_clean.dropna().unique():
    df[f"race_{r}"] = (race_clean == r).astype(int)
df["race_Missing"] = race_clean.isna().astype(int)
df["is_male"] = (df["gender"] == "Male").astype(int)
df["is_female"] = (df["gender"] == "Female").astype(int)

# Multiclass target: 0=NO, 1=>30, 2=<30
df_model = df.copy()
readmit_mapping = {'NO': 0, '>30': 1, '<30': 2}
df_model['readmitted'] = df_model['readmitted'].astype(str).str.strip()
df_model['readmit_class'] = df_model['readmitted'].map(readmit_mapping)

joblib.dump(df_model, os.path.join(BASE, "df_model.joblib"))
print("df_model shape:", df_model.shape)
print(df_model['readmit_class'].value_counts(normalize=True).sort_index().round(3))

## 2. Stage 2 - train/test split & preprocessor
70/30 stratified split (seed 10) - identical rows to the other models.

In [ ]:
target = 'readmit_class'
exclude_from_features = ['readmitted','readmit_class','race','gender','age','max_glu_serum',
    'A1Cresult','admission_type_id','discharge_disposition_id','admission_source_id',
    'repaglinide','nateglinide','chlorpropamide','acetohexamide','glyburide','tolbutamide',
    'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide','glyburide-metformin',
    'glipizide-metformin','glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone']

features = [c for c in df_model.columns if c not in exclude_from_features]
X = df_model[features].copy()
y = df_model[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=10, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Class distribution (train):")
print(y_train.value_counts(normalize=True).sort_index().round(3))

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

## 3. Import CTAB-GAN

In [ ]:
import sklearn.mixture as _mix

def _relax(cls):
    class _Relaxed(cls):
        def __init__(self, n_components=1, **kwargs):
            super().__init__(n_components=n_components, **kwargs)
    _Relaxed.__name__ = cls.__name__
    _Relaxed.__qualname__ = cls.__qualname__
    return _Relaxed

_mix.BayesianGaussianMixture = _relax(_mix.BayesianGaussianMixture)
_mix.GaussianMixture = _relax(_mix.GaussianMixture)

# Import CTAB-GAN
sys.path.append(CTABGAN_REPO)
from model.ctabgan import CTABGAN

## 4. Mitigation 1 - drop race features & rebuild the preprocessor

In [ ]:
SENSITIVE_PREFIXES = ("race_",)   # add "is_male","is_female" to also blind gender
sensitive_feats = [c for c in X_train.columns if c.startswith(SENSITIVE_PREFIXES)]
print("Dropping sensitive model features:", sensitive_feats)

X_train = X_train.drop(columns=sensitive_feats)
X_test = X_test.drop(columns=sensitive_feats)
numeric_cols = [c for c in numeric_cols if c not in sensitive_feats]
categorical_cols = [c for c in categorical_cols if c not in sensitive_feats]

preprocess = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                            ('scaler', StandardScaler())]), numeric_cols),
    ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                            ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])

## 5. Mitigation 2 - CTAB-GAN stratified augmentation
**Slow step**

In [ ]:
# Fit one GAN on the training frame including a race label column, then draw
# synthetic rows per race x class stratum to top each race group to 1:1:1.
# The race label never enters the model.

TARGET = "readmit_class"
RACE_COL = "race_audit"
EPOCHS = 150          # GAN epochs
MAX_GEN_ROUNDS = 30

GAN_CACHE = os.path.join(BASE, "gan_augmented.joblib")
if os.path.exists(GAN_CACHE):
    _d = joblib.load(GAN_CACHE)
    X_train_sm     = _d["X_train_sm"]
    y_train_sm     = _d["y_train_sm"]
    y_train_sm_ge1 = _d["y_train_sm_ge1"]
    y_train_sm_ge2 = _d["y_train_sm_ge2"]
    print("Loaded cached GAN augmentation from Drive:", X_train_sm.shape)
else:

    X_imp = X_train.copy()
    if X_imp[numeric_cols].isna().any().any():
        X_imp[numeric_cols] = SimpleImputer(strategy="median").fit_transform(X_imp[numeric_cols])
    if categorical_cols and X_imp[categorical_cols].isna().any().any():
        X_imp[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(X_imp[categorical_cols])

    train_race = df_model.loc[X_train.index, "race"].fillna("Missing").astype(str)
    train_df = X_imp.copy()
    train_df[RACE_COL] = train_race.values
    train_df[TARGET] = y_train.values

    binary_numeric = [c for c in numeric_cols if train_df[c].dropna().nunique() <= 2]
    continuous_numeric = [c for c in numeric_cols if c not in binary_numeric]
    integer_columns = [c for c in continuous_numeric if pd.api.types.is_integer_dtype(train_df[c])]
    ctab_categorical = list(categorical_cols) + binary_numeric + [RACE_COL, TARGET]

    raw_csv = os.path.join(BASE, "ctabgan_fair_train.csv")
    train_df.to_csv(raw_csv, index=False)

    synthesizer = CTABGAN(raw_csv_path=raw_csv, test_ratio=0.001,
        categorical_columns=ctab_categorical, log_columns=[], mixed_columns={},
        integer_columns=integer_columns, problem_type={"Classification": TARGET}, epochs=EPOCHS)
    print("Fitting CTAB-GAN ...")
    synthesizer.fit()

    strata = pd.crosstab(train_race, y_train)
    need = {}
    for g in strata.index:
        group_max = int(strata.loc[g].max())
        for k in [0, 1, 2]:
            n_gk = int(strata.loc[g].get(k, 0))
            if n_gk > 0:
                need[(g, k)] = group_max - n_gk
            elif group_max > 0:
                print(f"stratum race={g}, class={k} has no real rows - left empty")
    print("Synthetic rows needed per race x class stratum:")
    for (g, k), v in sorted(need.items()):
        if v > 0:
            print(f"  race={g:<20} class={k}: {v}")

    synth_parts = []
    for _ in range(MAX_GEN_ROUNDS):
        if all(v <= 0 for v in need.values()):
            break
        syn = synthesizer.generate_samples()
        syn[TARGET] = pd.to_numeric(syn[TARGET], errors="coerce").round()
        syn = syn.dropna(subset=[TARGET]); syn[TARGET] = syn[TARGET].astype(int)
        syn[RACE_COL] = syn[RACE_COL].astype(str)
        for (g, k), v in need.items():
            if v > 0:
                take = syn[(syn[RACE_COL] == g) & (syn[TARGET] == k)].head(v)
                if len(take):
                    synth_parts.append(take); need[(g, k)] -= len(take)
    if any(v > 0 for v in need.values()):
        print("WARNING: generation cap reached, still short:", {f"{g}/{k}": v for (g, k), v in need.items() if v > 0})

    synth_df = pd.concat(synth_parts, ignore_index=True)
    X_synth = synth_df[list(X_train.columns)].copy()
    for c in numeric_cols:
        X_synth[c] = pd.to_numeric(X_synth[c], errors="coerce").fillna(X_imp[c].median())
    for c in categorical_cols:
        X_synth[c] = X_synth[c].astype(str)
    y_synth = synth_df[TARGET].astype(int)

    X_train_sm = pd.concat([X_imp, X_synth], ignore_index=True)
    y_train_sm = pd.concat([y_train.reset_index(drop=True), y_synth], ignore_index=True)

    print("Original train shape: ", X_train.shape)
    print("Augmented train shape:", X_train_sm.shape)
    print("Original class distribution:\n", y_train.value_counts(normalize=True).sort_index().round(3))
    print("Augmented class distribution:\n", y_train_sm.value_counts(normalize=True).sort_index().round(3))

    y_train_sm_ge1 = (y_train_sm >= 1).astype(int)
    y_train_sm_ge2 = (y_train_sm >= 2).astype(int)
    joblib.dump({"X_train_sm": X_train_sm, "y_train_sm": y_train_sm,
                 "y_train_sm_ge1": y_train_sm_ge1, "y_train_sm_ge2": y_train_sm_ge2}, GAN_CACHE)
    print("Saved GAN augmentation to Drive ->", GAN_CACHE)

## 6-11. Per-seed model (looped over all seeds)

In [ ]:
param_dist_xgb = {
    "clf__n_estimators": randint(200, 700),
    "clf__max_depth": randint(3, 8),
    "clf__learning_rate": loguniform(0.01, 0.3),
    "clf__subsample": uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__min_child_weight": randint(1, 10),
    "clf__reg_lambda": loguniform(1e-2, 10),
}
REFERENCE_RACE = "Caucasian"
CLASSES = {0: "NO", 1: ">30", 2: "<30"}
TEAL = {"NO": "#80CBC4", ">30": "#26A69A", "<30": "#00695C"}
OFFS = {"NO": 0.24, ">30": 0.0, "<30": -0.24}


def run_seed(SEED):
    ART = os.path.join(BASE, f"seed_{SEED}")
    os.makedirs(ART, exist_ok=True)
    done_flag = os.path.join(ART, "DONE")
    if os.path.exists(done_flag):
        print(f"  seed {SEED} already complete -> skipping")
        return
    print(f"\n{'='*60}\n  SEED {SEED}  ->  {ART}\n{'='*60}")

    # Train the two cumulative heads (reseeded)
    def train_head(y_bin, scoring, label):
        pos = (y_bin == 1).sum(); neg = (y_bin == 0).sum(); spw = neg / pos
        pipe = Pipeline([("preprocess", clone(preprocess)),
            ("clf", XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                scale_pos_weight=spw, n_jobs=-1, random_state=SEED, tree_method="hist"))])
        search = RandomizedSearchCV(pipe, param_dist_xgb, n_iter=40, scoring=scoring, n_jobs=-1,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
            verbose=0, random_state=SEED)
        search.fit(X_train_sm, y_bin)
        print(f"  {label} best CV: {search.best_score_:.4f}")
        return search.best_estimator_

    best_ge1 = train_head(y_train_sm_ge1, "roc_auc", "Model 1 (y>=1)")
    best_ge2 = train_head(y_train_sm_ge2, "average_precision", "Model 2 (y>=2)")

    # Isotonic calibration
    cal_ge1 = CalibratedClassifierCV(clone(best_ge1), method="isotonic", cv=5).fit(X_train_sm, y_train_sm_ge1)
    cal_ge2 = CalibratedClassifierCV(clone(best_ge2), method="isotonic", cv=5).fit(X_train_sm, y_train_sm_ge2)

# Predict + threshold decode
p_ge1_test = cal_ge1.predict_proba(X_test)[:, 1]
p_ge2_test = cal_ge2.predict_proba(X_test)[:, 1]
y_proba_ordinal = np.clip(np.vstack([1 - p_ge1_test, p_ge1_test - p_ge2_test, p_ge2_test]).T, 0, None)
y_proba_ordinal = y_proba_ordinal / y_proba_ordinal.sum(axis=1, keepdims=True)
y_pred_ordinal = y_proba_ordinal.argmax(axis=1)

y_true = y_test.values

# The target rates come from the training labels,
# so no held-out label informs the operating point.

rate_ge1 = (y_train >= 1).mean()
rate_ge2 = (y_train == 2).mean()

t1 = float(np.quantile(p_ge1_test, 1 - rate_ge1))
t2 = float(np.quantile(p_ge2_test, 1 - rate_ge2))
print(f"thresholds (test-ranked, train-rate)   t1 (>=1): {t1:.3f}   t2 (<30): {t2:.3f}")

y_pred_3way = np.where(p_ge2_test >= t2, 2, np.where(p_ge1_test >= t1, 1, 0))

cm_3way = pd.DataFrame(confusion_matrix(y_true, y_pred_3way),
                       index=["NO", ">30", "<30"], columns=["NO", ">30", "<30"])
cm_3way.to_csv(os.path.join(ART, "confusion_matrix_3way_ctabgan_fair.csv"))
print("  AUC(OVR):", round(roc_auc_score(y_test, y_proba_ordinal, multi_class="ovr", average="weighted"), 3),
      "| QWK:", round(cohen_kappa_score(y_true, y_pred_3way, weights="quadratic"), 3))
joblib.dump({"p_ge1_test": p_ge1_test, "p_ge2_test": p_ge2_test,
    "y_proba_ordinal": y_proba_ordinal, "y_pred_ordinal": y_pred_ordinal,
    "y_pred_3way": y_pred_3way, "t1": t1, "t2": t2},
    os.path.join(ART, "predictions_ctabgan_fair.joblib"))

    # Three-way fairness audits
    class_names = {0: "NO", 1: ">30", 2: "<30"}
    age_bucket = pd.cut(df_model.loc[X_test.index, "age_numeric"],
                        bins=[0, 20, 40, 60, 90, 100],
                        labels=["10-20", "20-40", "40-60", "60-90", "90-100"],
                        right=False, include_lowest=True)
    audit = pd.DataFrame({
        "race": df_model.loc[X_test.index, "race"].fillna("Missing").astype(str).values,
        "gender": df_model.loc[X_test.index, "gender"].astype(str).values,
        "age": age_bucket.values, "y_true": y_true, "y_pred": y_pred_3way})
    audit["race_age"] = audit["race"] + "_" + audit["age"].astype(str)

    def class_metrics(sub, cls):
        yt = (sub["y_true"] == cls).astype(int).values
        yp = (sub["y_pred"] == cls).astype(int).values
        tp = int(((yp==1)&(yt==1)).sum()); fp = int(((yp==1)&(yt==0)).sum())
        tn = int(((yp==0)&(yt==0)).sum()); fn = int(((yp==0)&(yt==1)).sum())
        P, N, n = tp+fn, tn+fp, len(sub)
        d = lambda a, b: a/b if b else np.nan
        return dict(n=n, base_rate=d(P, n), sel_rate=d(tp+fp, n), flag_ratio=d(d(tp+fp, n), d(P, n)),
                    TPR=d(tp, P), FPR=d(fp, N), PPV=d(tp, tp+fp))

    for attr in ["race", "gender", "age", "race_age"]:
        rows = []
        for g, sub in audit.groupby(attr, observed=True):
            if len(sub) < 50:
                continue
            for cls in [0, 1, 2]:
                m = class_metrics(sub, cls); m[attr] = g; m["class"] = class_names[cls]; rows.append(m)
        cols = [attr, "class", "n", "base_rate", "sel_rate", "flag_ratio", "TPR", "FPR", "PPV"]
        pd.DataFrame(rows)[cols].to_csv(os.path.join(ART, f"fairness_{attr}_ctabgan_fair.csv"), index=False)

    # Logits + per-seed forest plot
    d_logit = pd.DataFrame({"race": audit["race"].values})
    results = {}
    for k, name in CLASSES.items():
        d_logit["y"] = (y_pred_3way == k).astype(int)
        res = smf.logit(f'y ~ C(race, Treatment(reference="{REFERENCE_RACE}"))', data=d_logit).fit(disp=0)
        conf = res.conf_int()
        t = pd.DataFrame({"term": res.params.index, "odds_ratio": np.exp(res.params.values),
            "or_lo": np.exp(conf[0].values), "or_hi": np.exp(conf[1].values)})
        t = t[t["term"].str.contains(r"\[T\.")].copy()
        t["label"] = t["term"].str.replace(r'.*\[T\.(.+)\]', r'\1', regex=True)
        results[name] = t.set_index("label")

    race_counts = d_logit["race"].value_counts()
    races = [r for r in race_counts.index if r != REFERENCE_RACE and r in results["<30"].index]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axvspan(0.8, 1.25, color="grey", alpha=0.08, zorder=0)
    for yi in range(len(races)):
        ax.axhline(yi, color="0.9", lw=0.8, zorder=0)
    for name in CLASSES.values():
        t = results[name].reindex(races)
        for yi, (_, row) in enumerate(t.iterrows()):
            yy = yi + OFFS[name]
            ax.plot([row["or_lo"], row["or_hi"]], [yy, yy], color=TEAL[name], lw=2.4,
                    solid_capstyle="round", zorder=2)
            ax.plot(row["odds_ratio"], yy, "o", color=TEAL[name], ms=6.5,
                    markeredgecolor="white", markeredgewidth=0.8, zorder=3)
    ax.axvline(1.0, color="black", ls="--", lw=1, zorder=1)
    ax.set_xscale("log"); ax.set_xlim(0.2, 5.0)
    ax.xaxis.set_major_locator(FixedLocator([0.2, 0.5, 1.0, 2.0, 5.0]))
    ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_locator(NullLocator())
    ax.set_yticks(range(len(races))); ax.set_yticklabels(races); ax.set_ylim(-0.6, len(races) - 0.4)
    ax.set_xlabel("Odds ratio vs Caucasian (log scale)")
    ax.set_title(f"Race allocation bias (fair CTAB-GAN, seed {SEED})")
    ax.grid(axis="x", ls=":", alpha=0.4)
    ax.legend(handles=[Line2D([0], [0], color=TEAL[n], lw=2.4, marker="o", label=n) for n in CLASSES.values()],
              title="Predicted outcome", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(ART, "logit_forest_3way_ctabgan_fair.png"), dpi=200, bbox_inches="tight")
    plt.close(fig)
    open(done_flag, "w").write("ok")   # mark this seed complete (resume skips it)
    print(f"  saved + marked DONE -> {ART}")


In [ ]:
# Run every seed

for _s in SEEDS:
    run_seed(_s)
print("\nALL SEEDS DONE.")


## 12. Aggregate seeds - mean forest plot, confusion matrix, fairness

In [ ]:
# Aggregate seeds - mean forest plot (mean point, averaged CI whiskers)
MEAN = os.path.join(BASE, "mean"); os.makedirs(MEAN, exist_ok=True)
race_all = df_model.loc[X_test.index, "race"].fillna("Missing").astype(str).values
seed_dirs = sorted(glob.glob(os.path.join(BASE, "seed_*")))

frames, n_seeds = [], 0
for d in seed_dirs:
    pp = os.path.join(d, "predictions_ctabgan_fair.joblib")
    if not os.path.exists(pp):
        continue
    n_seeds += 1
    yp = np.asarray(joblib.load(pp)["y_pred_3way"])
    dd = pd.DataFrame({"race": race_all})
    for k, name in CLASSES.items():
        dd["y"] = (yp == k).astype(int)
        if dd["y"].nunique() < 2:
            continue
        res = smf.logit(f'y ~ C(race, Treatment(reference="{REFERENCE_RACE}"))', data=dd).fit(disp=0)
        conf = res.conf_int()
        t = pd.DataFrame({"term": res.params.index, "odds_ratio": np.exp(res.params.values),
            "or_lo": np.exp(conf[0].values), "or_hi": np.exp(conf[1].values)})
        t = t[t["term"].str.contains(r"\[T\.")].copy()
        t["race"] = t["term"].str.replace(r'.*\[T\.(.+)\]', r'\1', regex=True); t["class"] = name
        frames.append(t[["class", "race", "odds_ratio", "or_lo", "or_hi"]])

allf = pd.concat(frames, ignore_index=True)
mean_or = allf.groupby(["class", "race"], observed=True)[["odds_ratio", "or_lo", "or_hi"]].mean().reset_index()
sd_or = allf.groupby(["class", "race"], observed=True)["odds_ratio"].std().reset_index().rename(
    columns={"odds_ratio": "or_sd_across_seeds"})
mean_or.merge(sd_or, on=["class", "race"]).round(4).to_csv(
    os.path.join(MEAN, "forest_or_mean_ctabgan_fair.csv"), index=False)

race_counts = pd.Series(race_all).value_counts()
races = [r for r in race_counts.index if r != REFERENCE_RACE and r in set(mean_or["race"])]
fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(0.8, 1.25, color="grey", alpha=0.08, zorder=0)
for yi in range(len(races)):
    ax.axhline(yi, color="0.9", lw=0.8, zorder=0)
for name in CLASSES.values():
    sub = mean_or[mean_or["class"] == name].set_index("race").reindex(races)
    for yi, (_, row) in enumerate(sub.iterrows()):
        if pd.isna(row["odds_ratio"]):
            continue
        yy = yi + OFFS[name]
        ax.plot([row["or_lo"], row["or_hi"]], [yy, yy], color=TEAL[name], lw=2.4,
                solid_capstyle="round", zorder=2)
        ax.plot(row["odds_ratio"], yy, "o", color=TEAL[name], ms=6.5,
                markeredgecolor="white", markeredgewidth=0.8, zorder=3)
ax.axvline(1.0, color="black", ls="--", lw=1, zorder=1)
ax.set_xscale("log"); ax.set_xlim(0.2, 5.0)
ax.xaxis.set_major_locator(FixedLocator([0.2, 0.5, 1.0, 2.0, 5.0]))
ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_locator(NullLocator())
ax.set_yticks(range(len(races))); ax.set_yticklabels(races); ax.set_ylim(-0.6, len(races) - 0.4)
ax.set_xlabel("Odds ratio vs Caucasian (log scale)")
ax.set_title(f"Race allocation bias (fair CTAB-GAN, mean of {n_seeds} seeds)")
ax.grid(axis="x", ls=":", alpha=0.4)
ax.legend(handles=[Line2D([0], [0], color=TEAL[n], lw=2.4, marker="o", label=n) for n in CLASSES.values()],
          title="Predicted outcome", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(MEAN, "logit_forest_3way_ctabgan_fair_mean.png"), dpi=200, bbox_inches="tight")
plt.show()

# mean confusion matrix + mean fairness tables
cms = [pd.read_csv(os.path.join(d, "confusion_matrix_3way_ctabgan_fair.csv"), index_col=0)
       for d in seed_dirs if os.path.exists(os.path.join(d, "confusion_matrix_3way_ctabgan_fair.csv"))]
if cms:
    (sum(cms) / len(cms)).round(2).to_csv(os.path.join(MEAN, "confusion_matrix_3way_ctabgan_fair_mean.csv"))
for attr in ["race", "gender", "age", "race_age"]:
    fs = [pd.read_csv(os.path.join(d, f"fairness_{attr}_ctabgan_fair.csv"))
          for d in seed_dirs if os.path.exists(os.path.join(d, f"fairness_{attr}_ctabgan_fair.csv"))]
    if not fs:
        continue
    cat = pd.concat(fs, ignore_index=True); num = cat.select_dtypes("number").columns.tolist()
    cat.groupby([attr, "class"], observed=True)[num].mean().reset_index().round(4).to_csv(
        os.path.join(MEAN, f"fairness_{attr}_ctabgan_fair_mean.csv"), index=False)
print(f"mean artifacts ({n_seeds} seeds) ->", MEAN)
